In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
bronze_table = "observatorio_dev.bronze.niveles_embalses"
silver_table = "observatorio_dev.silver.niveles_embalses"

In [0]:
bronze_df = spark.table(bronze_table)

bronze_count = bronze_df.count()

assert bronze_count > 0, "La tabla bronze.niveles_embalses está vacía."

print("Registros en Bronze niveles_embalses:", bronze_count)

In [0]:
expected_columns = [
    "id",
    "codigo_variable",
    "fecha_inicio",
    "codigo_duracion",
    "unidad_medida",
    "codigo_planta",
    "version",
    "valor",
    "source_file_name",
    "source_file_path",
    "ingestion_timestamp",
    "load_date"
]

missing_columns = [
    column_name for column_name in expected_columns
    if column_name not in bronze_df.columns
]

assert not missing_columns, (
    f"Faltan columnas en Bronze niveles_embalses: {missing_columns}"
)

print("Validación OK: Bronze niveles_embalses tiene las columnas esperadas.")

In [0]:
silver_df = (
    bronze_df
    .select(
        F.col("id"),

        F.upper(F.trim(F.col("codigo_variable"))).alias("codigo_variable"),

        F.to_timestamp(F.col("fecha_inicio")).alias("fecha_inicio"),

        F.upper(F.trim(F.col("codigo_duracion"))).alias("codigo_duracion"),
        F.upper(F.trim(F.col("unidad_medida"))).alias("unidad_medida"),
        F.upper(F.trim(F.col("codigo_planta"))).alias("codigo_planta"),
        F.upper(F.trim(F.col("version"))).alias("version"),

        F.regexp_replace(F.col("valor"), ",", ".").cast("double").alias("valor"),

        F.col("source_file_name"),
        F.col("source_file_path"),
        F.col("ingestion_timestamp"),
        F.col("load_date"),

        F.current_timestamp().alias("silver_created_at"),
        F.current_timestamp().alias("silver_updated_at")
    )
)

In [0]:
missing_keys_count = (
    silver_df
    .filter(
        F.col("codigo_variable").isNull() |
        F.col("fecha_inicio").isNull() |
        F.col("codigo_planta").isNull() |
        F.col("version").isNull()
    )
    .count()
)

assert missing_keys_count == 0, (
    f"Hay {missing_keys_count} registros sin codigo_variable, fecha_inicio, codigo_planta o version."
)

print("Validación OK: no hay registros sin llaves mínimas.")

In [0]:
invalid_value_count = (
    silver_df
    .filter(F.col("valor").isNull())
    .count()
)

print("Registros con valor nulo después de convertir a DOUBLE:", invalid_value_count)

In [0]:
duplicates_df = (
    silver_df
    .groupBy(
        "codigo_variable",
        "fecha_inicio",
        "codigo_planta",
        "version"
    )
    .count()
    .filter(F.col("count") > 1)
)

duplicates_count = duplicates_df.count()

print("Duplicados antes de deduplicar:", duplicates_count)

display(duplicates_df)

In [0]:
window_spec = (
    Window
    .partitionBy(
        "codigo_variable",
        "fecha_inicio",
        "codigo_planta",
        "version"
    )
    .orderBy(
        F.col("load_date").desc_nulls_last(),
        F.col("ingestion_timestamp").desc_nulls_last(),
        F.col("id").desc_nulls_last()
    )
)

silver_df = (
    silver_df
    .withColumn("row_number", F.row_number().over(window_spec))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

print("Registros después de deduplicar:", silver_df.count())

In [0]:
new_records_count = silver_df.count()

print("Registros a procesar:", new_records_count)

if new_records_count == 0:
    print("No hay registros para cargar en Silver.")

else:
    target = DeltaTable.forName(spark, silver_table)

    (
        target.alias("target")
        .merge(
            silver_df.alias("source"),
            """
            target.codigo_variable = source.codigo_variable
            AND target.fecha_inicio = source.fecha_inicio
            AND target.codigo_planta = source.codigo_planta
            AND target.version = source.version
            """
        )
        .whenMatchedUpdate(set={
            "id": "source.id",
            "codigo_duracion": "source.codigo_duracion",
            "unidad_medida": "source.unidad_medida",
            "valor": "source.valor",
            "source_file_name": "source.source_file_name",
            "source_file_path": "source.source_file_path",
            "ingestion_timestamp": "source.ingestion_timestamp",
            "load_date": "source.load_date",
            "silver_updated_at": "source.silver_updated_at"
        })
        .whenNotMatchedInsert(values={
            "id": "source.id",
            "codigo_variable": "source.codigo_variable",
            "fecha_inicio": "source.fecha_inicio",
            "codigo_duracion": "source.codigo_duracion",
            "unidad_medida": "source.unidad_medida",
            "codigo_planta": "source.codigo_planta",
            "version": "source.version",
            "valor": "source.valor",
            "source_file_name": "source.source_file_name",
            "source_file_path": "source.source_file_path",
            "ingestion_timestamp": "source.ingestion_timestamp",
            "load_date": "source.load_date",
            "silver_created_at": "source.silver_created_at",
            "silver_updated_at": "source.silver_updated_at"
        })
        .execute()
    )

    print("MERGE de niveles_embalses ejecutado correctamente.")

In [0]:
silver_count = spark.table(silver_table).count()

assert silver_count > 0, (
    "La tabla silver.niveles_embalses quedó vacía después del MERGE."
)

print("Registros actuales en silver.niveles_embalses:", silver_count)